# Notebook 06A — Model C Extended-Training Probe

**Status:** exploratory follow-on to the frozen Notebook 05 A/B/C experiment.

This notebook asks one narrow question: **was Model C still training-duration constrained at epoch 3 / update 3,663, or does continued training reach validation saturation or begin to overfit?**

The original A/B/C comparison is immutable. The frozen Model C point remains validation loss **3.684501**, perplexity **39.83**, at update **3,663**. Nothing produced here may replace that point in Notebook 05.

## 1. D-095 — Experimental contract

The extension resumes the **exact resumable Model C production checkpoint** at epoch 3 / optimizer update 3,663. It preserves the tokenizer, exact 20M-token corpus, architecture, context/packing, effective batch semantics, optimizer parameter groups, dropout, gradient clipping, validation split/procedure, and numerical semantics.

Only two things change relative to the frozen experiment: (1) training is allowed to continue beyond three epochs, and (2) because the original cosine schedule has ended, the extension uses the terminal learning rate **2e-4 as a constant LR** rather than restarting or retuning the schedule.

Validation remains every 200 optimizer updates plus epoch end. Training loss and validation loss/perplexity must be retained over time in a separate 06A artifact namespace.

**Early stopping:** minimize validation loss, `min_delta=0.001`, `patience=6` consecutive non-improving validation observations. A single increase never stops training.

**Safety ceiling:** at most 10 additional epochs. If validation is still improving at the ceiling, the conclusion is *no saturation observed within the tested range*.

The detailed canonical decision is recorded in `docs/decisions/06a_model_c_extended_training_probe.md`.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class ExtendedTrainingContract:
    start_epoch: int = 3
    start_update: int = 3663
    frozen_val_loss: float = 3.684501
    frozen_val_ppl: float = 39.83
    model_parameters: int = 33_497_600
    context_length: int = 512
    examples_per_epoch: int = 39_062
    scored_targets_per_epoch: int = 19_999_744
    updates_per_epoch: int = 1_221
    effective_batch_targets: int = 16_384
    extension_lr: float = 2e-4
    validation_every_updates: int = 200
    validation_targets: int = 256_512
    min_delta: float = 0.001
    patience: int = 6
    max_additional_epochs: int = 10

CONTRACT = ExtendedTrainingContract()
CONTRACT

In [ ]:
# D-095 internal consistency gate — no training or checkpoint mutation occurs here.
max_additional_updates = CONTRACT.max_additional_epochs * CONTRACT.updates_per_epoch
max_global_update = CONTRACT.start_update + max_additional_updates

assert CONTRACT.start_update == 3 * CONTRACT.updates_per_epoch
assert max_additional_updates == 12_210
assert max_global_update == 15_873
assert CONTRACT.extension_lr == 2e-4
assert CONTRACT.patience > 1, 'Early stopping must not stop on the first increase.'
assert CONTRACT.min_delta > 0
assert CONTRACT.validation_targets == 256_512
assert CONTRACT.model_parameters == 33_497_600

print('D-095 contract arithmetic: PASS')
print(f'Frozen start: epoch {CONTRACT.start_epoch}, update {CONTRACT.start_update:,}')
print(f'Extension ceiling: +{CONTRACT.max_additional_epochs} epochs / +{max_additional_updates:,} updates')
print(f'Max global update: {max_global_update:,}')
print(f'Early stopping: min_delta={CONTRACT.min_delta}, patience={CONTRACT.patience}')
print('Frozen A/B/C comparison mutation: NOT PERMITTED')

## Pause point

D-095 defines and internally validates the extension contract. **No extension training has started.**

The next chunk (D-096) should implement the hard resume/provenance gate: locate the persisted Model C resumable checkpoint, verify its update/epoch/model/optimizer/scaler/RNG/config identity, reconstruct and hash-gate the canonical train/validation streams, and refuse to continue if any required state is missing or incompatible.